In [2]:
import os 
import pandas as pd 
import numpy as np

from dotenv import load_dotenv

import requests 
import json 

TO DO:

- get list of air quality sensors (Golemio)
- get list of microclimate sensors (Golemio)
- get list of Weather stations (CHMI)

- join this to one dataset mapping the relevent locations together

- download relevant datasources, joint them based on mapping defined above 

- do analysis (...)

In [ ]:
#data chmi

#https://opendata.chmi.cz/meteorology/climate/historical/data/1hour/

#kbely wsi 0-20000-0-11567

## this gets ony one specific station !!!


In [20]:
chmi_url = 'https://opendata.chmi.cz/'
chmi_route = '/meteorology/climate/historical/data/1hour/2020/1h-0-20000-0-11567-202003.json'

chmi_headers = {
    'accept': 'application/json',
    'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
}

chmi_params = { 
            'from': '2020-03-13T10:54:00.000Z', 
            'to': '2020-03-15T13:05:00.000Z'
}

chmi_response = requests.get(f'{chmi_url}{chmi_route}', headers = chmi_headers, timeout = 60)

data_chmi = chmi_response.json()

In [21]:
print(type(data_chmi))
print(data_chmi.keys())
print(data_chmi['data'].keys())
print(data_chmi['data']['data'].keys())

print(data_chmi)

<class 'dict'>
dict_keys(['zaznamID', 'datovyZdrojID', 'datovyTokID', 'datumVytvoreni', 'verzeDat', 'data'])
dict_keys(['type', 'data'])
dict_keys(['header', 'values'])
{'zaznamID': 'c18f83fa-7366-e0de-5964-d0071f90d830', 'datovyZdrojID': 'meteorologie', 'datovyTokID': 'Open.Data.1H', 'datumVytvoreni': '2025-05-23T12:52:02.082Z', 'verzeDat': '1.0', 'data': {'type': 'DataCollection', 'data': {'header': 'STATION,ELEMENT,DT,VAL,FLAG,QUALITY', 'values': [['0-20000-0-11567', 'E', '2020-03-01T00:00:00Z', 7.3, '', 0.0], ['0-20000-0-11567', 'E', '2020-03-01T01:00:00Z', 8.2, '', 0.0], ['0-20000-0-11567', 'E', '2020-03-01T02:00:00Z', 8.3, '', 0.0], ['0-20000-0-11567', 'E', '2020-03-01T03:00:00Z', 7.7, '', 0.0], ['0-20000-0-11567', 'E', '2020-03-01T04:00:00Z', 7.6, '', 0.0], ['0-20000-0-11567', 'E', '2020-03-01T05:00:00Z', 7.3, '', 0.0], ['0-20000-0-11567', 'E', '2020-03-01T06:00:00Z', 7.0, '', 0.0], ['0-20000-0-11567', 'E', '2020-03-01T07:00:00Z', 6.7, '', 0.0], ['0-20000-0-11567', 'E', '2020-03

In [22]:
data_chmi_sub = data_chmi['data']['data']
chmi_headers = data_chmi_sub['header'].split(',')
chmi_values = data_chmi_sub['values']

df_chmi = pd.DataFrame(chmi_values, columns=chmi_headers)

df_chmi.head()

,STATION,ELEMENT,DT,VAL,FLAG,QUALITY
0,0-20000-0-11567,E,2020-03-01T00:00:00Z,7.3,,0.0
1,0-20000-0-11567,E,2020-03-01T01:00:00Z,8.2,,0.0
2,0-20000-0-11567,E,2020-03-01T02:00:00Z,8.3,,0.0
3,0-20000-0-11567,E,2020-03-01T03:00:00Z,7.7,,0.0
4,0-20000-0-11567,E,2020-03-01T04:00:00Z,7.6,,0.0


In [ ]:
### golemio data 


##### air quality

## metadata: https://opendata.chmi.cz//air_quality/recent/metadata/metadata.json 

In [3]:
load_dotenv('api_key.env')
api_key = os.getenv('GOLEMIO_API_KEY')

print(api_key is not None)

True


In [4]:
api_url = 'https://api.golemio.cz/'
route = '/v2/airqualitystations'

headers = {
    'X-access-token': api_key, 
    'accept': 'application/json',
    'User-Agent': 'JEM207 DataProcessingCourse (Educational access; contact: 19658413@fsv.cuni.cz)'
}

params = {

}

response = requests.get(f'{api_url}{route}', headers = headers, timeout = 60)

data = response.json()

In [5]:
print(type(data))

print(data.keys())

<class 'dict'>
dict_keys(['features', 'type'])


In [6]:
print(json.dumps(data, indent=2, ensure_ascii = False))

{
  "features": [
    {
      "geometry": {
        "coordinates": [
          14.380116,
          50.084385
        ],
        "type": "Point"
      },
      "properties": {
        "measurement": {
          "AQ_hourly_index": "1B",
          "components": [
            {
              "averaged_time": {
                "averaged_hours": "3",
                "value": 36.3
              },
              "type": "NO2"
            },
            {
              "averaged_time": {
                "averaged_hours": "3",
                "value": 30.5
              },
              "type": "PM10"
            }
          ]
        },
        "id": "ABREA",
        "name": "Praha 6-Břevnov",
        "updated_at": "2026-04-27T08:40:00.896Z",
        "district": "praha-6"
      },
      "type": "Feature"
    },
    {
      "geometry": {
        "coordinates": [
          14.44365,
          50.108845
        ],
        "type": "Point"
      },
      "properties": {
        "measurement": {
   

In [7]:
def print_json_structure(obj, indent=0):
    pad = "  " * indent

    if isinstance(obj, dict):
        for key, value in obj.items():
            print(f"{pad}{key}: {type(value).__name__}")
            print_json_structure(value, indent + 1)

    elif isinstance(obj, list):
        print(f"{pad}[list] len={len(obj)}")
        if obj:
            print_json_structure(obj[0], indent + 1)

print_json_structure(data)

features: list
  [list] len=17
    geometry: dict
      coordinates: list
        [list] len=2
      type: str
    properties: dict
      measurement: dict
        AQ_hourly_index: str
        components: list
          [list] len=2
            averaged_time: dict
              averaged_hours: str
              value: float
            type: str
      id: str
      name: str
      updated_at: str
      district: str
    type: str
type: str


In [82]:
features = data['features']
df = pd.json_normalize(features)

In [83]:
tmp = df.explode("properties.measurement.components", ignore_index=True)
dirs = pd.json_normalize(
    tmp["properties.measurement.components"]
).add_prefix("properties.measurement.components.")

df = pd.concat([tmp.drop(columns=["properties.measurement.components"]), dirs], axis=1)

df.head()

,type,geometry.coordinates,geometry.type,properties.measurement.AQ_hourly_index,properties.id,properties.name,properties.updated_at,properties.district,properties.measurement.components.type,properties.measurement.components.averaged_time.averaged_hours,properties.measurement.components.averaged_time.value
0,Feature,"[14.380116, 50.084385]",Point,1B,ABREA,Praha 6-Břevnov,2026-04-27T08:40:00.896Z,praha-6,NO2,3,36.3
1,Feature,"[14.380116, 50.084385]",Point,1B,ABREA,Praha 6-Břevnov,2026-04-27T08:40:00.896Z,praha-6,PM10,3,30.5
2,Feature,"[14.44365, 50.108845]",Point,1B,AHOLA,Praha 7-Holešovice,2026-04-27T08:40:00.896Z,praha-7,NO2,3,42.0
3,Feature,"[14.44365, 50.108845]",Point,1B,AHOLA,Praha 7-Holešovice,2026-04-27T08:40:00.896Z,praha-7,PM10,3,38.7
4,Feature,"[14.44365, 50.108845]",Point,1B,AHOLA,Praha 7-Holešovice,2026-04-27T08:40:00.896Z,praha-7,PM2_5,3,23.0


In [84]:
station_cols = {
    'geometry.coordinates': 'coordinates', 
    'properties.id': 'id', 
    'properties.name': 'name', 
    'properties.district': 'district', 
    'properties.measurement.components.type': 'components'
}

In [85]:
air_quality_stations = df[station_cols.keys()]

air_quality_stations = air_quality_stations.rename(columns=station_cols)

In [88]:
air_quality_stations.head()

,id,coordinates,name,district,components
0,ABREA,"[14.380116, 50.084385]",Praha 6-Břevnov,praha-6,"[NO2, PM10]"
1,ACHOA,"[14.51745, 50.03017]",Praha 4-Chodov,praha-11,"[NO2, PM10]"
2,AHOLA,"[14.44365, 50.108845]",Praha 7-Holešovice,praha-7,"[NO2, PM10, PM2_5]"
3,AKALA,"[14.442049, 50.094238]",Praha 8-Karlín,praha-8,"[NO2, PM10]"
4,AKOBA,"[14.467578, 50.122189]",Praha 8-Kobylisy,praha-8,"[NO2, O3, PM10]"


In [87]:
air_quality_stations = (
    air_quality_stations.groupby('id', as_index=False)
    .agg({
        'coordinates': 'first',
        'name': 'first',
        'district': 'first',
        'components': list
    })
)

In [ ]:
#### golemio

### microclimate sensors

In [ ]:
# TODO if necessary